# Challenge 3 — Data Science & AI/ML-Model Evaluation, Benchmarking & Selection Research & Consulting Hub

**Aluno(a): Fabiana Rotella Campanari**

Conteúdo das **Aulas 01 a 09**, até a validação cruzada aninhada (*nested CV*).

Rode cada célula e **escreva sua resposta** no campo indicado. Você **não** precisa escrever
código — o objetivo é interpretar e comparar os resultados. Os arquivos `.csv` estão nesta pasta.

> **Regra deste desafio:** toda afirmação que menciona um valor deve **trazer o valor**,
> copiado da saída da sua célula. Os números mudam de caderno para caderno.

Algumas células rodam várias buscas e podem levar um ou dois minutos. Espere o `[*]` virar
número antes de passar para a próxima.


In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
from scipy.stats import uniform, loguniform, randint
from sklearn.model_selection import (train_test_split, cross_val_score, cross_val_predict,
                                     StratifiedKFold, GridSearchCV, RandomizedSearchCV)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score)

def logistica():
    """Logistica dentro de um Pipeline (Aula 05)."""
    return Pipeline([("esc", StandardScaler()),
                     ("clf", LogisticRegression(max_iter=5000))])

def arvore():
    return DecisionTreeClassifier(max_depth=2, random_state=0)

def floresta():
    return RandomForestClassifier(n_estimators=50, random_state=0, n_jobs=-1)

print("ambiente pronto")

ambiente pronto


## Exercício 1 — Grade e sorteio com o mesmo orçamento  *(1,5 pontos)*

Dois hiperparâmetros da floresta, 16 medições para cada caminho: uma grade 4 × 4 e uma `RandomizedSearchCV` com `n_iter = 16`. Depois, a mesma busca aleatória repetida com 6 sementes diferentes.

**(a)** Qual caminho deu a melhor nota, e por quanto? Dê os dois valores.

**(b)** Quantos valores **distintos** de `max_features` cada caminho testou? Ligue esse número à diferença de nota.

**(c)** Olhe as 6 sementes: quanto o resultado do sorteio balança? Cite a média, o mínimo, o máximo e a fração que ficou acima da grade. Dá para declarar um vencedor comparando um número com outro?

**(d)** Se você acrescentasse um terceiro hiperparâmetro com 4 valores, quantos ajustes cada caminho passaria a custar? Mostre as duas contas.

In [ ]:
df = pd.read_csv("q_sorteio.csv")
X = df.drop(columns="alvo").values
y = df["alvo"].values
cv = StratifiedKFold(5, shuffle=True, random_state=0)

grade = {"max_features": [0.2, 0.4, 0.6, 0.8],
         "min_samples_leaf": [1, 5, 10, 20]}
g = GridSearchCV(floresta(), grade, cv=cv, scoring="accuracy", n_jobs=-1).fit(X, y)
print("grade  : 16 medicoes | melhor", round(g.best_score_, 4), "|", g.best_params_)

dist = {"max_features": uniform(0.1, 0.8),
        "min_samples_leaf": randint(1, 21)}
r = RandomizedSearchCV(floresta(), dist, n_iter=16, cv=cv, scoring="accuracy",
                       random_state=0, n_jobs=-1).fit(X, y)
print("sorteio: 16 medicoes | melhor", round(r.best_score_, 4), "|",
      {k: (round(v, 4) if isinstance(v, float) else v) for k, v in r.best_params_.items()})

print("\nvalores distintos de max_features testados:")
print("  grade  :", len(set(grade["max_features"])))
print("  sorteio:", len(set(np.round(r.cv_results_["param_max_features"].data.astype(float), 8))))

notas = np.array([RandomizedSearchCV(floresta(), dist, n_iter=16, cv=cv,
                                     scoring="accuracy", random_state=s,
                                     n_jobs=-1).fit(X, y).best_score_
                  for s in range(6)])
print(f"\n6 sementes: media {notas.mean():.4f} | min {notas.min():.4f}"
      f" | max {notas.max():.4f}")
print(f"fracao acima da grade ({g.best_score_:.4f}): {np.mean(notas >= g.best_score_):.0%}")

grade  : 16 medicoes | melhor 0.6355 | {'max_features': 0.6, 'min_samples_leaf': 20}
sorteio: 16 medicoes | melhor 0.6355 | {'max_features': np.float64(0.5164), 'min_samples_leaf': 20}

valores distintos de max_features testados:
  grade  : 4
  sorteio: 16

6 sementes: media 0.6335 | min 0.6235 | max 0.6474
fracao acima da grade (0.6355): 33%


**Resposta:**

**(a)** Ambos os métodos obtiveram exatamente o mesmo resultado! A acurácia obtida pela busca em grade foi de \( 0.6355 \) e a acurácia obtida pelo sorteio aleatório também foi de exatamente \( 0.6355 \). A diferença absoluta entre as notas de ambos os caminhos é de \( 0.0000 \).

<br><br>

**(b)** A busca em grade avaliou somente \( 4 \) valores distintos para o parâmetro `max_features` (repetindo as combinações dentro do seu espaço fixo), ao passo que a busca aleatória por sorteio avaliou \( 16 \) valores totalmente distintos para `max_features`. Apesar da busca aleatória cobrir um espaço amostral de parâmetros quatro vezes maior de forma bem distribuída, a nota final acabou empatada em \( 0.6355 \) porque a influência crucial de desempenho estava atrelada ao critério limitador `min_samples_leaf = 20` (encontrado em ambas).

<br><br>

**(c)** O sorteio apresenta flutuações consideráveis dependendo do valor da semente aleatória adotada: a média de acurácia observada nas 6 sementes foi de \( 0.6335 \), apresentando uma nota mínima de \( 0.6235 \) e uma nota máxima de \( 0.6474 \). A fração de sementes que superou ou igualou a nota da grade de \( 0.6355 \) foi de apenas \( 33\% \) (apenas duas sementes de seis no total). Com esses números, fica evidente que não é correto rotular o sorteio como vencedor definitivo, uma vez que, dependendo da semente de amostragem inicial, ele corre um risco considerável de desempenhar abaixo da busca em grade.

<br><br><br>

**(d)** Se a gente colocar mais um parâmetro de \( 4 \) opções possíveis no teste:

O custo da busca em grade sofreria um impacto de produto cartesiano, passando a custar: <br>


<br>

$$ 4 \times 4 \times 4 \times 5 = 320 $$

<br>

treinamentos e avaliações de modelos:

O custo do sorteio continuaria estritamente idêntico, totalizando:<br>


<br>

$$ 16 \times 5 = 80 $$

<br>

rodadas de treinamento, uma vez que o limite superior `n_iter = 16` é definido e controlado de forma rígida pelo analista.

──────────────────────── ⋆⋅☆⋅⋆ ────────────────────────

## Exercício 2 — Escolher e medir no mesmo conjunto  *(1,5 pontos)*

A base `q_escolha.csv` é cortada em treino e teste. O primeiro bloco varre sete valores de `C` **medindo cada um no teste** e fica com o melhor. O segundo bloco escolhe o `C` por validação cruzada **dentro do treino** e toca o teste uma única vez, no fim.

**(a)** Qual `C` e qual nota saem de cada um dos dois caminhos? Dê os valores e a diferença entre as duas notas finais.

**(b)** Por que a nota do primeiro caminho não é o desempenho do modelo? Use o tamanho do teste na sua explicação.

**(c)** Qual é a regra que o segundo caminho respeita, e quando o teste pode ser tocado?

In [ ]:
df = pd.read_csv("q_escolha.csv")
X = df.drop(columns="alvo").values
y = df["alvo"].values
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, stratify=y,
                                          random_state=0)
print("treino:", len(y_tr), "| teste:", len(y_te))

valores = [0.001, 0.01, 0.1, 1, 10, 100, 1000]

# CAMINHO 1 — escolher o C olhando o TESTE
no_teste = {}
for C in valores:
    m = logistica().set_params(clf__C=C).fit(X_tr, y_tr)
    no_teste[C] = m.score(X_te, y_te)
print("\nnotas NO TESTE, por C:")
for C, s in no_teste.items():
    print(f"  C = {C:<7} -> {s:.4f}")
melhor_C = max(no_teste, key=no_teste.get)
print(f"melhor olhando o teste: C = {melhor_C} com {no_teste[melhor_C]:.4f}")

# CAMINHO 2 — escolher por CV no treino e medir uma vez so
cv = StratifiedKFold(5, shuffle=True, random_state=0)
g = GridSearchCV(logistica(), {"clf__C": valores}, cv=cv, n_jobs=-1).fit(X_tr, y_tr)
honesta = logistica().set_params(clf__C=g.best_params_["clf__C"]).fit(X_tr, y_tr).score(X_te, y_te)
C_cv = g.best_params_["clf__C"]
print(f"\nescolhido por CV no treino: C = {C_cv} (nota de CV {g.best_score_:.4f})")
print(f"nota no teste, medida uma vez: {honesta:.4f}")
print(f"diferenca para o caminho 1: {no_teste[melhor_C] - honesta:+.4f}")

treino: 106 | teste: 46

notas NO TESTE, por C:
  C = 0.001   -> 0.6739
  C = 0.01    -> 0.6522
  C = 0.1     -> 0.6304
  C = 1       -> 0.6087
  C = 10      -> 0.6304
  C = 100     -> 0.6304
  C = 1000    -> 0.6304
melhor olhando o teste: C = 0.001 com 0.6739

escolhido por CV no treino: C = 0.1 (nota de CV 0.6887)
nota no teste, medida uma vez: 0.6304
diferenca para o caminho 1: +0.0435


**Resposta:**

**(a)** No primeiro caminho, em que o hiperparâmetro foi escolhido monitorando diretamente o desempenho no conjunto de teste, o melhor valor obtido foi \( C = 0.001 \) com uma acurácia de teste correspondente de \( 0.6739 \). Já no segundo caminho (com escolha blindada de \( C \) via validação cruzada no treino), o melhor valor selecionado foi \( C = 0.1 \) (que obteve \( 0.6887 \) de acurácia média no treino) e que resultou em uma acurácia real final no teste de \( 0.6304 \). A diferença entre as acurácias obtidas nos dois caminhos de teste é de \( +0.0435 \) a favor da primeira abordagem.


<br><br>

**(b)** A nota de \( 0.6739 \) do primeiro cenário é completamente ilusória e enviesada porque o conjunto de teste foi utilizado ativamente na escolha da configuração do estimador (ocorrendo o chamado *data leakage*). Em um conjunto de teste muito reduzido, contendo apenas \( 46 \) amostras de dados, pequenas variações estatísticas de acerto devido à escolha do hiperparâmetro podem inflar as métricas de forma espúria e artificial, mascarando a real capacidade do classificador de generalizar para dados nunca vistos.

<br><br>

**(c)** O segundo método cumpre rigorosamente o princípio fundamental de blindagem de dados, impedindo o vazamento de informações para fora do treino. Sob este paradigma padrão, o conjunto de teste só deve ser consultado uma única vez ao término de toda a etapa experimental para estimar a acurácia de generalização final do modelo com hiperparâmetros já fixados.

──────────────────────── ⋆⋅☆⋅⋆ ────────────────────────

## Exercício 3 — A métrica escolhe o modelo  *(1,5 pontos)*

A base `q_metrica.csv` é desbalanceada. Dois modelos — uma logística e uma árvore rasa — são avaliados com a MESMA validação cruzada estratificada, em três métricas: acurácia, F1 e average precision (AP). A última célula mostra também o que um modelo que nunca diz 1 conseguiria.

**(a)** Qual modelo vence em **acurácia** e qual vence em **F1**? Dê os quatro valores.

**(b)** A acurácia do modelo "sempre 0" é alta? Compare-a com a do melhor modelo e explique o que isso revela sobre a acurácia numa base desbalanceada.

**(c)** O que o **F1** mede que a acurácia não mede? Use a precisão e a revocação impressas para explicar por que o F1 do vencedor em acurácia é o que é.

**(d)** Qual modelo você entregaria, e com qual métrica justificaria a escolha? Cite os números.

In [ ]:
df = pd.read_csv("q_metrica.csv")
X = df.drop(columns="alvo").values
y = df["alvo"].values
print("linhas:", len(y), "| positivos:", int(y.sum()), f"({y.mean():.1%})")

cv = StratifiedKFold(5, shuffle=True, random_state=0)
modelos = {"logistica": logistica(), "arvore": arvore()}

for nome, m in modelos.items():
    p = cross_val_predict(m, X, y, cv=cv)
    print(f"\n{nome}")
    print(f"  acuracia : {accuracy_score(y, p):.4f}")
    print(f"  precisao : {precision_score(y, p, zero_division=0):.4f}")
    print(f"  revocacao: {recall_score(y, p, zero_division=0):.4f}")
    print(f"  F1       : {f1_score(y, p, zero_division=0):.4f}")
    print(f"  AP       : {cross_val_score(m, X, y, cv=cv, scoring='average_precision').mean():.4f}")

zero = np.zeros_like(y)
print(f"\nmodelo que NUNCA diz 1 -> acuracia {accuracy_score(y, zero):.4f}"
      f" | F1 {f1_score(y, zero, zero_division=0):.4f}")

linhas: 502 | positivos: 38 (7.6%)

logistica
  acuracia : 0.9223
  precisao : 0.4286
  revocacao: 0.0789
  F1       : 0.1333
  AP       : 0.3399

arvore
  acuracia : 0.9203
  precisao : 0.4286
  revocacao: 0.1579
  F1       : 0.2308
  AP       : 0.2008

modelo que NUNCA diz 1 -> acuracia 0.9243 | F1 0.0000


**Resposta:**

**(a)** Se avaliarmos unicamente a métrica de **acurácia**, a regressão logística vence a disputa atingindo a marca de \( 0.9223 \) contra \( 0.9203 \) observada na árvore de decisão rasa. Contudo, na avaliação por **F1-score**, a árvore de decisão vence o confronto com \( 0.2308 \) em comparação com o baixo valor de \( 0.1333 \) obtido pela regressão logística.

<br>



**(b)** Sim, a acurácia obtida pelo classificador estático baseline (que prevê somente a classe majoritária 0) é de \( 0.9243 \), valor este ligeiramente superior tanto à acurácia da logística (\( 0.9223 \)) quanto à da árvore (\( 0.9203 \)). Em cenários nos quais os dados apresentam forte desbalanceamento — como neste caso, que possui apenas \( 38 \) instâncias positivas de um conjunto de \( 502 \) exemplos (ou seja, apenas \( 7.6\% \) positivos) —, a acurácia mascara severamente a ineficiência do classificador, pois predizer cegamente a classe majoritária garante uma nota alta sem que nenhuma detecção da classe minoritária de interesse seja realmente efetuada.


<br>

**(c)** O F1-score quantifica de forma harmônica a relação entre a Precisão e a Revocação (Recall), penalizando modelos que ignoram uma classe. A regressão logística atingiu uma acurácia elevada de \( 0.9223 \) porque quase nunca fez predições para a classe positiva (classe 1), o que resultou em uma precisão de \( 0.4286 \) combinada a uma revocação extremamente baixa de apenas \( 0.0789 \) (detectando pouquíssimos positivos). Essa ausência de sensibilidade derrubou seu F1-score final para apenas \( 0.1333 \).

<br>

**(d)** Eu entregaria o modelo baseado em **regressão logística** justificando a escolha através da métrica de **Average Precision (AP)**, onde a regressão logística atingiu a marca de \( 0.3399 \) em comparação aos \( 0.2008 \) obtidos pela árvore de decisão. O AP é a métrica mais robusta para conjuntos desbalanceados por analisar o desempenho de ordenação do modelo através de múltiplos limiares de decisão (curva precisão-revocação), sem ficar engessado ao limiar fixo padrão de \( 0.5 \).

──────────────────────── ⋆⋅☆⋅⋆ ────────────────────────


## Exercício 4 — A grade e a borda da faixa  *(2,0 pontos)*

Uma `GridSearchCV` varre o `C` da logística numa faixa estreita. O código imprime a tabela inteira do `cv_results_` e, depois, a mesma busca com a faixa ampliada.

**(a)** Qual `C` venceu na primeira grade, e qual foi a nota? Onde esse valor está dentro da faixa que foi varrida?

**(b)** O que um vencedor na **borda** da faixa indica? Diga o que você faria a seguir.

**(c)** Depois de ampliar a faixa, qual `C` venceu e qual foi a nota? A primeira grade estava certa ou errada? Explique.

**(d)** Quantos ajustes de modelo cada uma das duas grades custou? Mostre a conta.

In [ ]:
df = pd.read_csv("q_grade.csv")
X = df.drop(columns="alvo").values
y = df["alvo"].values
cv = StratifiedKFold(5, shuffle=True, random_state=0)

def busca_grade(valores):
    g = GridSearchCV(logistica(), {"clf__C": valores}, cv=cv,
                     scoring="accuracy", n_jobs=-1).fit(X, y)
    t = pd.DataFrame({"C": valores,
                      "media": g.cv_results_["mean_test_score"].round(4),
                      "desvio": g.cv_results_["std_test_score"].round(4)})
    return g, t

faixa1 = [0.0001, 0.001, 0.01, 0.1]
g1, t1 = busca_grade(faixa1)
print("GRADE 1 — faixa de 0,0001 a 0,1")
print(t1.to_string(index=False))
print("melhor C:", g1.best_params_["clf__C"], "| nota:", round(g1.best_score_, 4))

faixa2 = [0.0001, 0.001, 0.01, 0.1, 1, 10, 100, 1000]
g2, t2 = busca_grade(faixa2)
print("\nGRADE 2 — faixa ampliada ate 1000")
print(t2.to_string(index=False))
print("melhor C:", g2.best_params_["clf__C"], "| nota:", round(g2.best_score_, 4))

GRADE 1 — faixa de 0,0001 a 0,1
     C  media  desvio
0.0001 0.8255  0.1083
0.0010 0.9117  0.0308
0.0100 0.9177  0.0246
0.1000 0.9359  0.0262
melhor C: 0.1 | nota: 0.9359

GRADE 2 — faixa ampliada ate 1000
        C  media  desvio
   0.0001 0.8255  0.1083
   0.0010 0.9117  0.0308
   0.0100 0.9177  0.0246
   0.1000 0.9359  0.0262
   1.0000 0.9696  0.0234
  10.0000 0.9786  0.0155
 100.0000 0.9939  0.0075
1000.0000 0.9908  0.0123
melhor C: 100 | nota: 0.9939


**Resposta:**

**(a)** Na primeira grade proposta, o hiperparâmetro vencedor foi o valor limite de \( C = 0.1 \), alcançando a acurácia média de \( 0.9359 \). Este valor situa-se na extremidade direita absoluta da faixa inicial varrida, que era restrita ao intervalo \( [0.0001, 0.001, 0.01, 0.1] \).

<br><br>

**(b)** Quando o melhor hiperparâmetro selecionado localiza-se exatamente na borda do intervalo de busca, há um forte indício de que valores ainda mais extremos na mesma direção podem gerar resultados de desempenho superiores. O procedimento correto a seguir consiste em expandir a grade de busca criando um intervalo mais amplo de testes naquela direção.


<br><br>

**(c)** Após expandir a busca incluindo ordens de magnitude superiores, o valor ideal obtido foi \( C = 100.0 \), elevando a acurácia média para \( 0.9939 \). Portanto, conclui-se que a primeira parametrização de busca em grade estava conceitualmente errada e subotimizada, pois a restrição inicial impediu o classificador de explorar as regiões de melhor ajuste.


<br><br><br>

**(d)** A primeira rotina de ajuste custou um total de \( 20 \) treinamentos de modelos decorrente do cálculo de:

<br>

$$ 4 \text{ valores de } C \times 5 \text{ dobras} = 20 $$

<br>

A segunda rotina estendida custou um total de \( 40 \) treinamentos de modelos decorrente do cálculo de:


<br>


$$ 8 \text{ valores de } C \times 5 \text{ dobras} = 40 $$

──────────────────────── ⋆⋅☆⋅⋆ ────────────────────────

## Exercício 5 — Os três números da mesma busca  *(2,0 pontos)*

Da mesma busca saem três números: o `best_score_`, a **nested CV** e a nota no **teste guardado**. O código produz os três e mostra as dobras de fora da nested, uma a uma.

**(a)** Dê os três números. Qual é o maior?

**(b)** Qual deles **não** pode ser reportado como desempenho do modelo, e por quê? (A resposta é de princípio, não de qual saiu maior.)

**(c)** Olhe as dobras de fora da nested, uma a uma, e o desvio. O que esse desvio diz sobre a diferença entre os três números?

**(d)** O que você escreveria no relatório final, em uma frase, usando os números do seu caderno?

In [ ]:
df = pd.read_csv("q_nested.csv")
X = df.drop(columns="alvo").values
y = df["alvo"].values

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, stratify=y,
                                          random_state=0)
print("treino:", len(y_tr), "| teste:", len(y_te), "| positivos no teste:", int(y_te.sum()))

dentro = StratifiedKFold(5, shuffle=True, random_state=0)
fora   = StratifiedKFold(5, shuffle=True, random_state=1)
espaco = {"clf__C": loguniform(0.001, 1000)}
busca = RandomizedSearchCV(logistica(), espaco, n_iter=20, cv=dentro,
                           scoring="accuracy", random_state=0, n_jobs=-1)

nested = cross_val_score(busca, X_tr, y_tr, cv=fora, scoring="accuracy")
ajustada = busca.fit(X_tr, y_tr)
teste = ajustada.score(X_te, y_te)

print(f"\n1) best_score_   : {ajustada.best_score_:.4f}")
print(f"2) nested CV     : {nested.mean():.4f} +- {nested.std():.4f}")
print(f"   por dobra     : {np.round(nested, 4)}")
print(f"3) teste guardado: {teste:.4f}")

treino: 140 | teste: 60 | positivos no teste: 23

1) best_score_   : 0.7214
2) nested CV     : 0.7357 +- 0.0892
   por dobra     : [0.7857 0.5714 0.7857 0.7143 0.8214]
3) teste guardado: 0.6000


**Resposta:**

**(a)** Os três valores coletados foram:
1. Acurácia máxima interna (`best_score_`): \( 0.7214 \)
2. Média do teste de validação cruzada aninhada (`nested CV`): \( 0.7357 \)
3. Acurácia do conjunto de teste guardado: \( 0.6000 \)

O maior valor dentre os três informados é a média da validação cruzada aninhada (`nested CV`), que registrou \( 0.7357 \).


<br><br>

**(b)** O indicador `best_score_` (\( 0.7214 \)) não deve ser divulgado como a capacidade de generalização estável do modelo. Este número representa o valor máximo otimizado dentro de múltiplos testes de hiperparâmetros realizados na busca interna, carregando consigo um viés otimista de seleção de variáveis que mascara o real desempenho do modelo diante de novos cenários.


<br><br>

**(c)** Analisando as cinco dobras do teste externo da validação aninhada, observamos marcas que variam amplamente entre \( 0.5714 \) e \( 0.8214 \) (especificamente: \( 0.7857 \), \( 0.5714 \), \( 0.7857 \), \( 0.7143 \) e \( 0.8214 \)), o que resulta em um elevado desvio padrão de \( 0.0892 \). Essa oscilação acentuada evidencia que amostragens reduzidas geram forte instabilidade de avaliação, o que justifica perfeitamente o fato de a acurácia do teste de controle guardado (\( 0.6000 \)) ter caído abaixo do esperado, uma vez que ela se encontra dentro do intervalo de flutuação estatística normal do modelo.


<br><br>

**(d)** "Pela estimativa obtida via validação cruzada aninhada, o desempenho estimado para nosso classificador é de \( 0.7357 \pm 0.0892 \) de acurácia, o que é estatisticamente compatível com a nota de \( 0.6000 \) registrada em nossa avaliação com o conjunto de teste isolado."

──────────────────────── ⋆⋅☆⋅⋆ ────────────────────────

## Exercício 6 — Pódio ou platô?  *(1,5 pontos)*

Uma busca varre o `C` da logística em 12 valores. O código imprime o pódio completo — média e desvio entre dobras de cada posição.

**(a)** Qual é a diferença entre o 1º e o 5º colocado? E qual é o desvio entre dobras do 1º? Dê os dois valores e compare-os.

**(b)** Isso é um pódio ou um platô? Justifique com os números.

**(c)** Qual configuração você entregaria, e por quê? (Dica: entre empatados, há um critério melhor que "ficou em primeiro".)

In [ ]:
df = pd.read_csv("q_plato.csv")
X = df.drop(columns="alvo").values
y = df["alvo"].values
cv = StratifiedKFold(5, shuffle=True, random_state=0)

valores = [0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1, 3, 10, 30, 100, 300]
g = GridSearchCV(logistica(), {"clf__C": valores}, cv=cv,
                 scoring="accuracy", n_jobs=-1).fit(X, y)
tab = pd.DataFrame({"C": valores,
                    "media": g.cv_results_["mean_test_score"],
                    "desvio": g.cv_results_["std_test_score"]})
tab = tab.sort_values("media", ascending=False).reset_index(drop=True)
tab.index = tab.index + 1
print(tab.round(4).to_string())

dif = tab["media"].iloc[0] - tab["media"].iloc[4]
print(f"\ndiferenca 1o - 5o: {dif:.4f}")
print(f"desvio entre dobras do 1o: {tab['desvio'].iloc[0]:.4f}")

          C   media  desvio
1     0.001  0.5688  0.0364
2     0.003  0.5625  0.0395
3     0.010  0.5562  0.0459
4     0.030  0.5500  0.0468
5     0.100  0.5438  0.0508
6     0.300  0.5375  0.0459
7     1.000  0.5375  0.0459
8     3.000  0.5375  0.0459
9    10.000  0.5375  0.0459
10   30.000  0.5375  0.0459
11  100.000  0.5375  0.0459
12  300.000  0.5375  0.0459

diferenca 1o - 5o: 0.0250
desvio entre dobras do 1o: 0.0364


Resposta:

**(a)** A diferença absoluta de acurácia média obtida entre o primeiro colocado da tabela (\( C = 0.001 \) com nota média de \( 0.5688 \)) e o quinto colocado (\( C = 0.1 \) com nota média de \( 0.5438 \)) é de apenas \( 0.0250 \). Por sua vez, o desvio padrão observado nas dobras de validação cruzada do primeiro colocado é de \( 0.0364 \). Comparando-os, a diferença entre as médias das primeiras posições (\( 0.0250 \)) é consideravelmente inferior à oscilação média captada nas partições internas de dados (\( 0.0364 \)).

<br><br>


**(b)** Este resultado se configura claramente como um **platô**. Dado que o desvio padrão de desempenho entre as dobras do modelo líder (\( 0.0364 \)) é maior que a variação de pontuação média observada entre as posições de 1 a 5 no pódio (\( 0.0250 \)), torna-se impossível afirmar com consistência estatística que o modelo posicionado no primeiro lugar é verdadeiramente superior aos modelos subsequentes.

<br><br>

**(c)** Eu entregaria a configuração com menor complexidade (ou seja, maior regularização), optando por \( C = 0.001 \) (que obteve média de \( 0.5688 \) e desvio de \( 0.0364 \)) ou uma próxima dentro do platô de empate estatístico, pois o critério de parcimônia indica que devemos preferir o modelo mais simples e robusto para evitar sobreajuste ao ruído amostral.

🛸๋*ੈ✩* 🔭✮☾𖤓.☘︎ ݁˖⁷⁷⁷ㅤ✮ ⋆ ˚｡𖦹 ⋆｡°✩ ✮ ⋆ ˚｡𖦹 ⋆｡°✩  *ੈ✩‧₊˚  🛸๋*ੈ✩* 🔭✮☾𖤓.☘︎ ݁˖⁷⁷⁷ㅤ✮ ⋆ ˚｡𖦹 ⋆｡°✩ ✮ ⋆ ˚｡𖦹 ⋆｡°✩  *ੈ✩‧₊˚  

### ⚡️